In [1]:
#IMPORT CODE

%matplotlib widget

import os
import geopandas as gpd
import pandas as pd
import folium
import rasterio as rio
import cartopy.crs as ccrs
im

#LOAD DATA

    #method of travel csv, no geo
method = pd.read_csv('London_data/method.csv', skiprows=6)
    #boundary files
boroughs = gpd.read_file('London_data/London_Boroughs.gpkg')
    #PTAL data
ptal = pd.read_csv('London_data/MSOA_aggregated_PTAL_stats_2023.csv')


NameError: name 'im' is not defined

In [ ]:
#PTAL grid - large file so use pyogrio (4 seconds)
ptalgrid = gpd.read_file('London_data/PTAL_2023_Grid_100mx100m_Data.geojson')

In [ ]:
#smaller PTAL grid shapefile
ptal_grid = gpd.read_file('London_data/PTAL_2023_Grid_100mx100m_Data.shp')

In [ ]:
#delete unnecessary rows

method = method.drop(range(34,50))

In [ ]:
#CRS--------------------------------------------------------------
#create a universal crs
print(boroughs.crs)
#it is epsg=27700
london_crs = ccrs.epsg(27700)

In [ ]:
#find common attributes of datasets

boroughs.head()


In [ ]:
method.head()

In [ ]:
#merge them by nmenonic and gss code
#lets add geometries to method after renamingmnemonic to gss_code

method = method.rename(index=str, columns={'mnemonic':'gss_code'})

In [ ]:
borough_method = boroughs.merge(method, on='gss_code')

In [ ]:
borough_method

In [ ]:
#clean up
borough_method = borough_method.drop(columns=['Area','sub_2011','hectares','nonld_area'])

In [ ]:
#create a new column for total public transport use
borough_method['pt_uptake'] = borough_method['%.1']+borough_method['%.2']+borough_method['%.3']


In [ ]:
#create map feature and update legend. Add popups for boroughs.
columns_to_show = ['name', 'pt_uptake']
popup = folium.features.GeoJsonPopup(fields=columns_to_show,
                                     aliases=['Borough', 'Percentage Uptake of Public Transport'],
                                     localize=True,
                                     labels=True,
                                     style='background-color: white;'
                                    )


lm = borough_method.explore(column='pt_uptake',
                            cmap='viridis',
                           legend_kwds={'caption': 'Percentage Uptake of Public Transport'})

columns_to_show = ['name', 'pt_uptake']
popup = folium.features.GeoJsonPopup(fields=columns_to_show,
                                     aliases=['Borough', 'Percentage Uptake of Public Transport'],
                                     localize=True,
                                     labels=True,
                                     style='background-color: white;'
                                    )


#add our pt accessibility data
    #PTAL grid - large file so use pyogrio


def invisible_geo(feature):
    '''this dictionary can be passed any time we add a feature
    that we do not want visualized on the shapefile'''
    return{
        'fillColor':'transparent',
        'color':'None',
        'weight':1,
        'fillOpacity':0}

folium.GeoJson(data=borough_method, style_function=invisible_geo,
               popup=popup).add_to(lm)

#adding to the folium feature adds a strong blue outline and facecolor feature. Go back and
#define a style_function argument to prevent this when using folium.add_to

lm

#

In [ ]:
ptalgrid.head()

In [ ]:
#clip ptal grid to bounds of london data
    ptalgrid = ptalgrid.to_crs(london_crs)

#use a spatial index as the file is large
    ptalgrid.sindex
    ptal_clipped = gpd.clip(ptalgrid, borough_method['geometry'])
pass

In [ ]:
borough_method.crs

In [ ]:
#introduce PTAL data
ptal.head()

In [ ]:
help(gpd.GeoDataFrame)

In [ ]:

ptal_clipped = gpd.clip(ptal_grid, borough_method['geometry'])



In [ ]:
#this method is still taking too long, so it is necessary to cut down further
#try combining the broughs into one shape
ptal_grid = ptal_grid.to_crs(london_crs)
london_mask = borough_method.dissolve()
ptal_clipped = gpd.clip(ptal_grid, london_mask)
pass

In [ ]:
#instead of relying on the grid, it may be fruitful to create a point geometry

In [ ]:
#create a function that returns a map based on percentage uptake of different forms of transport

def uptake_function (borough_method):
    print('What method of transport uptake, as a percentage, would you like to see?')
    print('Options:\n Train\n Underground\n Bike\n Car\n Walking\n Bus')
    method= {'Train':'%.2',
     'Underground':'%.1',
     'Bike':'%.6',
     'Car':'%.5',
     'Walking':'%.7',
     'Bus':'%.3'}
    column_input = input('your choice here:')
    if column_input in method:
        chosen_column = method[column_input]
        m = borough_method.explore(column=chosen_column,
                               cmap='viridis',
                               legend_kwds={'caption':f'Percentage uptake of {column_input}'})
        popup = folium.features.GeoJsonPopup(fields=['name', chosen_column],
                                     aliases=['Borough', f'Percentage Uptake of {column_input}'],
                                     localize=True,
                                     labels=True,
                                     style='background-color: white;'
                                    )
        folium.GeoJson(data=borough_method, style_function=invisible_geo,
               popup=popup).add_to(m)
        return m
    else:
        print('that is not a method')
        
#improve the function by creating a dictionary inside to avoid using the alphanumeric titles                  

In [ ]:
uptake_function(borough_method)

In [ ]:
lm

In [ ]:
borough_method

In [ ]:
#create a function that will return the borough that uses the form of transport the most
#wihtout set_index, the function may just list the row number, not the borough name


#create dictionary for inputs and column name
methods = {'train':'%.2',
            'underground':'%.1',
            'bike':'%.6',
            'car':'%.5',
            'walking':'%.7',
            'bus':'%.3'}


def most_popular(borough_method):

    '''this is a function of our dataset
    first, print an opening message for the user
    then, tell Python to await another input
    if the input matches one of the terms in our 'methods' object
    so that our final printed output is variable, define them.
    first, tell python to find user choice within methods
    second, tell python to find THIS function WITHIN our dataset, at it maximum
    finally, the computer is able to output each function as part of the final output'''
    
    print('Which mode are you interested in?')
    user_choice = input()
    if user_choice in methods:
        column_name = methods[user_choice]    #identifies column in dict (easy)
        highest_use_borough = borough_method[column_name].idxmax()    #identifies column in dataset
        pc_use = borough_method.loc[highest_use_borough, column_name]      #locates exact value for column_name
        print(f'{user_choice} is most popular in {highest_use_borough}')
        print(f'{pc_use} percent of people use {user_choice}')
    else:
        print('that is not a method')

most_popular(borough_method)


In [ ]:
#we can investigate the patterns associated with commuting further

In [ ]:
print('What method of transport uptake, as a percentage, would you like to see?')
print('Options:\n Train\n Underground\n Bike\n Car\n Walking\n Bus')
method = {'Train': '%.2',
          'Underground': '%.1',
          'Bike': '%.6',
          'Car': '%.5',
          'Walking': '%.7',
          'Bus': '%.3'}

column_input = input('your choice here:')
if column_input in method:
    chosen_column = method[column_input]
    m = borough_method.explore(column=chosen_column,
                               cmap='viridis',
                               legend_kwds={'caption': f'Percentage uptake of {column_input}'})
    popup = folium.features.GeoJsonPopup(fields=['name', chosen_column],
                                         aliases=['Borough', f'Percentage Uptake of {column_input}'],
                                         localize=True,
                                         labels=True,
                                         style='background-color: white;'
                                         )
    folium.GeoJson(data=borough_method, style_function=invisible_geo,
                   popup=popup).add_to(m)

    m

    column_name = method[column_input]  # identifies column in dict (easy)
    highest_use_borough = borough_method[column_name].idxmax()  # identifies column in dataset
    pc_use = borough_method.loc[highest_use_borough, column_name]  # locates exact value for column_name
    print(f'{column_input} is most popular in {highest_use_borough}')
    print(f'{pc_use} percent of people use {column_input}')

    

else:
    print('Please enter a valid choice')

In [ ]:
def uptake_borough():
    print('What method of transport uptake, as a percentage, would you like to see?')
    print('Options:\n Train\n Underground\n Bike\n Car\n Walking\n Bus')
    method = {'Train': '%.2',
              'Underground': '%.1',
              'Bike': '%.6',
              'Car': '%.5',
              'Walking': '%.7',
              'Bus': '%.3'}

    while True:
        print('type exit to exit')
        column_input = input('your choice here:')

        if column_input == 'exit':
            break
        if column_input in method:  # if input in dictionary: yes
            chosen_column = method[column_input]  # define chosen_column as input in dictionary

            m = borough_method.explore(column=chosen_column,  # map uses chosen column
                                       cmap='viridis',
                                       legend_kwds={'caption': f'Percentage uptake of {column_input}'})
            popup = folium.features.GeoJsonPopup(fields=['name', chosen_column],
                                                 aliases=['Borough', f'Uptake of {column_input}%'],
                                                 localize=True,
                                                 labels=True,
                                                 style='background-color: white;'
                                                 )
            folium.GeoJson(data=borough_method, style_function=invisible_geo,
                           popup=popup).add_to(m)
            highest_use = borough_method[chosen_column].idxmax() #identifies row with highest value
            highest_use_borough = borough_method.loc[highest_use]# identifies row
            pc_use = highest_use_borough[chosen_column]  # locates exact value for column_name
            print(f'{column_input} is most popular in {highest_use_borough["name"]}')
            print(f'{pc_use} percent of people use {column_input} in {highest_use_borough["name"]}')

            map2_html = 'map2.html'
            m.save(map2_html)

            webbrowser.open(map2_html)

            break

        else:
            print('Please enter a valid choice')



In [ ]:
uptake_borough()
